In [1]:
import pandas as pd
from rich import print

In [2]:
df = pd.read_csv(filepath_or_buffer="./data.csv")

### 0. Check df. Little EDA

In [3]:
df.head()

,project_type,project,job,question,assessor,dt,label
0,project_type_1,project_1,job_14008,question_3,assessor_53,2025-07-08,1
1,project_type_0,project_0,job_14004,question_4,assessor_55,2025-07-08,1
2,project_type_1,project_1,job_14020,question_2,assessor_14,2025-07-08,0
3,project_type_1,project_1,job_14021,question_0,assessor_6,2025-07-08,1
4,project_type_0,project_2,job_14032,question_2,assessor_47,2025-07-08,0


In [4]:
df.shape

(708, 7)

In [5]:
df.dtypes

project_type      str
project           str
job               str
question          str
assessor          str
dt                str
label           int64
dtype: object

In [6]:
class ColNms:
    PROJECT_TYPE = "project_type"
    PROJECT = "project"
    JOB = "job"
    QUESTION = "question"
    ASSESSOR = "assessor"
    DT = "dt"
    LABEL = "label"

In [7]:
df[ColNms.DT].dtype

<StringDtype(storage='python', na_value=nan)>

In [8]:
df.isna().sum()

project_type    0
project         0
job             0
question        0
assessor        0
dt              1
label           0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(1)

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   project_type  708 non-null    str  
 1   project       708 non-null    str  
 2   job           708 non-null    str  
 3   question      708 non-null    str  
 4   assessor      708 non-null    str  
 5   dt            707 non-null    str  
 6   label         708 non-null    int64
dtypes: int64(1), str(6)
memory usage: 38.8 KB


In [36]:
df.describe()

,dt,label
count,706,706.000000
mean,2025-07-08 00:02:02.379603,0.548159
min,2025-07-08 00:00:00,0.000000
25%,2025-07-08 00:00:00,0.000000
50%,2025-07-08 00:00:00,1.000000
75%,2025-07-08 00:00:00,1.000000
max,2025-07-09 00:00:00,1.000000
std,NaN,0.498028


### 1. Fix the dtypes

In [11]:
df[ColNms.DT] = pd.to_datetime(df[ColNms.DT])

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   project_type  708 non-null    str           
 1   project       708 non-null    str           
 2   job           708 non-null    str           
 3   question      708 non-null    str           
 4   assessor      708 non-null    str           
 5   dt            707 non-null    datetime64[us]
 6   label         708 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(5)
memory usage: 38.8 KB


### 2. Clean edge cases

In [13]:
df = df.dropna()

In [14]:
df.shape

(707, 7)

In [15]:
df = df.drop_duplicates()

In [16]:
df.shape

(706, 7)

We dropped the dupes which are exactly the same but there is a subtler case:

since we have a composite PK there might me the case that the same assessor labeled the same question twice with different labels.

In [17]:
pk = [ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION, ColNms.ASSESSOR]

In [18]:
conflicts = df.groupby(pk)[ColNms.LABEL]

In [19]:
conflicts.head()

0      1
1      1
2      0
3      1
4      0
      ..
703    0
704    0
705    1
706    1
707    1
Name: label, Length: 706, dtype: int64

In [20]:
conflicts = df.groupby(pk)[ColNms.LABEL].nunique()

In [21]:
conflicts.head(100)

project    job        question    assessor   
project_0  job_14000  question_0  assessor_24    1
                                  assessor_41    1
                                  assessor_49    1
                                  assessor_5     1
                                  assessor_56    1
                                                ..
           job_14004  question_1  assessor_10    1
                                  assessor_44    1
                                  assessor_46    1
                                  assessor_50    1
                                  assessor_8     1
Name: label, Length: 100, dtype: int64

In [22]:
conflicts[conflicts > 1]

Series([], Name: label, dtype: int64)

### 3. Q1: prepare the df in the format other teams can consume

In [23]:
df[ColNms.PROJECT].nunique()

3

how many questions per project?

In [24]:
df.groupby(ColNms.PROJECT)[ColNms.QUESTION].nunique()

project
project_0    5
project_1    5
project_2    5
Name: question, dtype: int64

how many assessors labeled the same question?

In [32]:
with pd.option_context("display.max_rows", None):
    df_questions = df.groupby([ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION])[
        ColNms.ASSESSOR
    ].nunique()
    print(df_questions[df_questions > 5])

project    job        question  
project_1  job_14010  question_1    6
Name: assessor, dtype: int64

In [34]:
df_questions.describe()

count    141.000000
mean       5.007092
std        0.084215
min        5.000000
25%        5.000000
50%        5.000000
75%        5.000000
max        6.000000
Name: assessor, dtype: float64

we see it is a majority vote. let's prepare a df with lables based on majority vote

In [38]:
votes = df.groupby([ColNms.PROJECT, ColNms.JOB, ColNms.QUESTION])[ColNms.LABEL].mean()

In [ ]:
majority = votes[
    votes >= 0.5
]  # here you failed: you selected the rows whose mean is more or = 0.5. you filtered out.

In [40]:
majority.head()

project    job        question  
project_0  job_14000  question_0    0.8
                      question_2    0.8
           job_14001  question_2    1.0
                      question_4    0.6
           job_14002  question_1    1.0
Name: label, dtype: float64

In [41]:
majority = votes >= 0.5

In [42]:
majority.head()

project    job        question  
project_0  job_14000  question_0     True
                      question_1    False
                      question_2     True
                      question_3    False
                      question_4    False
Name: label, dtype: bool

In [43]:
majority = (votes >= 0.5).astype(int)

In [44]:
majority.head()

project    job        question  
project_0  job_14000  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    0
Name: label, dtype: int64

In [46]:
with pd.option_context("display.max_rows", None):
    print(majority)

project    job        question  
project_0  job_14000  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    0
           job_14001  question_0    0
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    1
           job_14002  question_0    0
                      question_1    1
                      question_2    1
                      question_3    1
           job_14003  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14004  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
                      question_4    0
           job_14005  question_0    1
                      question_1    0
                      question_2    0
           job_14006  question_0    0
                      question_1    1
                      question_2    0
           job_14007  question_0    1
                      question_1    1
                      question_2    0
                      question_3    1
                      question_4    0
project_1  job_14008  question_0    1
                      question_1    0
                      question_2    1
                      question_3    1
                      question_4    1
           job_14009  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14010  question_0    0
                      question_1    0
                      question_2    1
           job_14011  question_0    0
                      question_1    1
                      question_2    0
           job_14012  question_0    1
                      question_1    1
                      question_2    0
                      question_3    0
           job_14013  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
           job_14014  question_0    1
                      question_1    0
                      question_2    1
           job_14015  question_0    1
                      question_1    0
                      question_2    1
                      question_3    0
                      question_4    1
           job_14016  question_0    0
                      question_1    0
                      question_2    1
                      question_3    1
           job_14017  question_0    1
                      question_1    1
                      question_2    1
                      question_3    1
                      question_4    0
           job_14018  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
           job_14019  question_0    1
                      question_1    1
                      question_2    1
                      question_3    0
           job_14020  question_0    1
                      question_1    1
                      question_2    0
           job_14021  question_0    1
                      question_1    0
                      question_2    1
                      question_3    1
                      question_4    0
project_2  job_14022  question_0    0
                      question_1    0
                      question_2    0
                      question_3    1
                      question_4    0
           job_14023  question_0    0
                      question_1    1
                      question_2    1
                      question_3    0
           job_14024  question_0    1
                      question_1    1
                      question_2    1
                      question_3    0
                      question_4    0
           job_

In [48]:
majority.reset_index()

,project,job,question,label
0,project_0,job_14000,question_0,1
1,project_0,job_14000,question_1,0
2,project_0,job_14000,question_2,1
3,project_0,job_14000,question_3,0
4,project_0,job_14000,question_4,0
...,...,...,...,...
136,project_2,job_14033,question_2,1
137,project_2,job_14033,question_3,1
138,project_2,job_14034,question_0,1
139,project_2,job_14034,question_1,0


In [49]:
majority.reset_index(name="majority_vote")

,project,job,question,majority_vote
0,project_0,job_14000,question_0,1
1,project_0,job_14000,question_1,0
2,project_0,job_14000,question_2,1
3,project_0,job_14000,question_3,0
4,project_0,job_14000,question_4,0
...,...,...,...,...
136,project_2,job_14033,question_2,1
137,project_2,job_14033,question_3,1
138,project_2,job_14034,question_0,1
139,project_2,job_14034,question_1,0


### 4. Assessors quality label